In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('/content/drive/MyDrive/ex6DataPrepared/final_data (1).csv')
df['date'] = pd.to_datetime(df['date'])
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(by=['our_input_addrs', 'date'])

# A. Biến động giá
price_df = df[['date', 'close']].drop_duplicates().sort_values('date')
price_df['price_change_10d'] = price_df['close'].pct_change(periods=10)
price_df['std_10d'] = price_df['close'].rolling(window=10).std()
df = df.merge(price_df[['date', 'price_change_10d', 'std_10d']], on='date', how='left')

# B. Phí so với thị trường
df['fee_vs_market'] = df['fee_rate'] / (df.groupby('date')['fee_rate'].transform('mean') + 1e-9)

# C. Z-score theo lịch sử từng ví
def calculate_zscore(x):
    if len(x) < 3:
        return 0
    return (x - x.mean()) / (x.std() + 1e-9)
df['vol_zscore'] = df.groupby('our_input_addrs')['net_change_for_our'].transform(calculate_zscore)

# D. Chuẩn hóa phí
from sklearn.preprocessing import RobustScaler
robust_scaler = RobustScaler(quantile_range=(5, 95))
df['fee_rate_norm'] = robust_scaler.fit_transform(df[['fee_rate']])
df_buy = df[df['net_change_for_our'] > 0].copy()
# Features cho clustering
cluster_features = [
    'price_change_10d',
    'fee_vs_market',
    'vol_zscore',
    'fee_rate_norm',
    'is_whale'
]

# Làm sạch dữ liệu
X_cluster = df_buy[cluster_features].fillna(0)
X_cluster = X_cluster.replace([np.inf, -np.inf], 0)

# Chuẩn hóa
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Phân cụm K-Means với 3 cụm
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_buy['cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Đánh giá phân cụm
sil_score = silhouette_score(X_cluster_scaled, df_buy['cluster'])
# Tính trung bình các features của từng cụm
cluster_profile = df_buy.groupby('cluster')[cluster_features].mean()
cluster_profile['score'] = (
    cluster_profile['price_change_10d'] / cluster_profile['price_change_10d'].max() * 0.5 +
    cluster_profile['fee_vs_market'] / cluster_profile['fee_vs_market'].max() * 0.3 +
    cluster_profile['vol_zscore'] / cluster_profile['vol_zscore'].max() * 0.2
)

# Sắp xếp và gán nhãn
cluster_profile = cluster_profile.sort_values('score', ascending=False)
cluster_mapping = {
    cluster_profile.index[0]: 2,  # FOMO cao
    cluster_profile.index[1]: 1,  # FOMO trung bình
    cluster_profile.index[2]: 0   # Không FOMO
}

df_buy['target'] = df_buy['cluster'].map(cluster_mapping)

# Thêm target vào DataFrame gốc
df['target'] = 0
df.loc[df_buy.index, 'target'] = df_buy['target']
features = [
    'price_change_10d', 'std_10d', 'fee_vs_market', 'vol_zscore',
    'fee_rate_norm', 'is_whale', 'is_consolidation',
    'open', 'high', 'low', 'close', 'volume', 'n_inputs', 'n_outputs'
]
df_labeled = df[df['target'] != 0].copy()  # Lấy cả FOMO
non_fomo_sample = df[df['target'] == 0].sample(n=len(df_labeled) * 2, random_state=42)  # Lấy gấp đôi non-FOMO
df_balanced = pd.concat([df_labeled, non_fomo_sample])

print(f"Dữ liệu cân bằng: {len(df_balanced):,} giao dịch")
print(f"  - Không FOMO: {len(df_balanced[df_balanced['target'] == 0]):,}")
print(f"  - FOMO TB: {len(df_balanced[df_balanced['target'] == 1]):,}")
print(f"  - FOMO Cao: {len(df_balanced[df_balanced['target'] == 2]):,}")

X = df_balanced[features].fillna(0).replace([np.inf, -np.inf], 0)
y = df_balanced['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# HUẤN LUYỆN XGBOOST
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_alpha=0.1,         # L1 regularization
    reg_lambda=1.0,        # L2 regularization
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Không FOMO', 'FOMO TB', 'FOMO Cao']))

Dữ liệu cân bằng: 27,102 giao dịch
  - Không FOMO: 18,068
  - FOMO TB: 646
  - FOMO Cao: 8,388
              precision    recall  f1-score   support

  Không FOMO       0.99      0.98      0.98      4517
     FOMO TB       0.88      0.92      0.90       162
    FOMO Cao       0.96      0.99      0.97      2097

    accuracy                           0.98      6776
   macro avg       0.94      0.96      0.95      6776
weighted avg       0.98      0.98      0.98      6776

